# Setup (run once)

Install the packages below. You also need a `.env` file in this folder containing:

```
OPENAI_API_KEY=sk-...
```

*Linux only:* run `sudo apt install libportaudio2` first, so the microphone recording works.

In [1]:
%pip install sounddevice numpy librosa matplotlib scipy openai python-dotenv ipykernel pydub

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 932.9 kB/s eta 0:00:01
   ------------------------------- -------- 0.8/1.0 MB 860.9 kB/s eta 0:00:01
   ------------------------------- -------- 0.8/1.0 MB 860.9 kB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 784.4 kB/s  0:00:01
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 1.5 MB/s eta 0:00:02
   ------- -------------------------------- 0.5/2.8 MB 1.5 MB/s eta 0:00:02
   ----------- ---------------------------- 0.8/2.8 MB 1.2 MB/s eta 0:00:02
   -------------- ------------------------- 1.0/2.8 MB 1.2 MB/s eta 0:00:02
   ------------------ --------------------- 1.3/2.8 MB 1.1 MB/s eta 0:00:02
   ------------------ -------------

In [1]:
from pathlib import Path
import os
import shutil
from dotenv import load_dotenv
from openai import OpenAI

project_dir = Path.cwd()
env_path = project_dir / ".env"

load_dotenv(env_path, override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not env_path.is_file():
    raise FileNotFoundError(".env was not found in the notebook folder.")

if not api_key or api_key == "YOUR_API_KEY":
    raise ValueError("Add your real API key to .env and save it.")

client = OpenAI(api_key=api_key)

for folder in ["audio", "chunks", "outputs"]:
    (project_dir / folder).mkdir(exist_ok=True)

print("Project folder:", project_dir)
print("API key loaded; client initialized.")
print("Folders ready: audio, chunks, outputs")
print("FFmpeg found:", shutil.which("ffmpeg") is not None)

Project folder: c:\Users\Roodabeh Seif\Downloads\Ironhack\Week1.Day5\lab-whisper-stt
API key loaded; client initialized.
Folders ready: audio, chunks, outputs
FFmpeg found: True


In [2]:
audio_files = [
    p for p in (project_dir / "audio").iterdir()
    if p.is_file()
]

for file in audio_files:
    print(file.name)

project_nimbus_meeting.mp3


In [3]:
from pydub import AudioSegment
from IPython.display import Audio, display

audio_path = project_dir / "audio" / "project_nimbus_meeting.mp3"
meeting_audio = AudioSegment.from_file(audio_path)

print("File:", audio_path.name)
print("Duration (seconds):", round(len(meeting_audio) / 1000, 2))
print("Channels:", meeting_audio.channels)
print("Sample rate:", meeting_audio.frame_rate)

short_audio = meeting_audio[:30_000]
short_path = project_dir / "chunks" / "sample_30_seconds.wav"

short_audio.export(short_path, format="wav")
display(Audio(filename=str(short_path)))

File: project_nimbus_meeting.mp3
Duration (seconds): 309.2
Channels: 1
Sample rate: 22050


In [4]:
with short_path.open("rb") as audio_file:
    basic_transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

print(basic_transcript.text)

output_path = project_dir / "outputs" / "basic_transcription.txt"
output_path.write_text(basic_transcript.text, encoding="utf-8")

print("\nSaved:", output_path.name)

Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication

Saved: basic_transcription.txt


In [5]:
context_prompt = (
    "Project Nimbus, weekly engineering sync, "
    "Kubernetes migration, PostgreSQL latency, "
    "Q3 OKRs, Priya, Marcus, authentication."
)

with short_path.open("rb") as audio_file:
    prompted_transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=context_prompt
    )

print("WITHOUT PROMPT:\n")
print(basic_transcript.text)

print("\nWITH PROMPT:\n")
print(prompted_transcript.text)

(project_dir / "outputs" / "prompted_transcription.txt").write_text(
    prompted_transcript.text,
    encoding="utf-8"
)

WITHOUT PROMPT:

Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication

WITH PROMPT:

Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication


371

In [6]:
chunk_length_ms = 60_000
chunk_info = []

for index, start_ms in enumerate(
    range(0, len(meeting_audio), chunk_length_ms)
):
    chunk = meeting_audio[start_ms:start_ms + chunk_length_ms]

    # Mono, 16 kHz, 16-bit WAV keeps each file small.
    chunk = chunk.set_channels(1).set_frame_rate(16000).set_sample_width(2)

    chunk_path = project_dir / "chunks" / f"chunk_{index:03d}.wav"
    chunk.export(chunk_path, format="wav")

    chunk_info.append({
        "path": chunk_path,
        "offset_seconds": start_ms / 1000
    })

    print(
        chunk_path.name,
        "| start:", start_ms / 1000,
        "| duration:", len(chunk) / 1000,
        "seconds"
    )

print("\nTotal chunks:", len(chunk_info))

chunk_000.wav | start: 0.0 | duration: 60.0 seconds
chunk_001.wav | start: 60.0 | duration: 60.0 seconds
chunk_002.wav | start: 120.0 | duration: 60.0 seconds
chunk_003.wav | start: 180.0 | duration: 60.0 seconds
chunk_004.wav | start: 240.0 | duration: 60.0 seconds
chunk_005.wav | start: 300.0 | duration: 9.202 seconds

Total chunks: 6


In [7]:
all_segments = []
chunk_results = []

for index, info in enumerate(chunk_info):
    print(f"Transcribing chunk {index + 1}/{len(chunk_info)}...")

    with info["path"].open("rb") as audio_file:
        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json",
            timestamp_granularities=["segment"]
        )

    data = result.model_dump()
    chunk_results.append(data)

    for segment in data.get("segments", []):
        all_segments.append({
            "start": segment["start"] + info["offset_seconds"],
            "end": segment["end"] + info["offset_seconds"],
            "text": segment["text"].strip()
        })

full_text = " ".join(segment["text"] for segment in all_segments)

print("\nCOMPLETE TRANSCRIPTION:\n")
print(full_text)

print("\nFIRST FIVE SEGMENTS:\n")
for segment in all_segments[:5]:
    print(
        f'[{segment["start"]:.2f}s - {segment["end"]:.2f}s] '
        f'{segment["text"]}'
    )

Transcribing chunk 1/6...
Transcribing chunk 2/6...
Transcribing chunk 3/6...
Transcribing chunk 4/6...
Transcribing chunk 5/6...
Transcribing chunk 6/6...

COMPLETE TRANSCRIPTION:

Good morning everyone! This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus! So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the Redis cache. When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class. I opened a ticket, Jira number NIM-482, and I expect it to be fixed by Tuesday. Okay, good. Do we need anything from the DevOps team? Yes. Yes, we need Thomas to approve the Terraform changes for the VPC peering. Wit

In [8]:
import json

def format_timestamp(seconds, separator="."):
    total_ms = round(seconds * 1000)
    hours, remainder = divmod(total_ms, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, milliseconds = divmod(remainder, 1000)
    return (
        f"{hours:02}:{minutes:02}:{secs:02}"
        f"{separator}{milliseconds:03}"
    )

output_dir = project_dir / "outputs"

# Readable text with timestamps
text_lines = [
    f'[{format_timestamp(s["start"])} - '
    f'{format_timestamp(s["end"])}] {s["text"]}'
    for s in all_segments
]

(output_dir / "transcription.txt").write_text(
    "\n".join(text_lines),
    encoding="utf-8"
)

# Subtitle file
srt_blocks = [
    f'{index}\n'
    f'{format_timestamp(s["start"], ",")} --> '
    f'{format_timestamp(s["end"], ",")}\n'
    f'{s["text"]}\n'
    for index, s in enumerate(all_segments, start=1)
]

(output_dir / "transcription.srt").write_text(
    "\n".join(srt_blocks),
    encoding="utf-8"
)

# Structured data
(output_dir / "transcription.json").write_text(
    json.dumps(
        {"text": full_text, "segments": all_segments},
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("Saved: transcription.txt")
print("Saved: transcription.srt")
print("Saved: transcription.json")
print("\nSubtitle preview:\n")
print("\n".join(srt_blocks[:2]))

Saved: transcription.txt
Saved: transcription.srt
Saved: transcription.json

Subtitle preview:

1
00:00:00,000 --> 00:00:05,800
Good morning everyone! This is the weekly engineering sync for Project Nimbus.

2
00:00:05,800 --> 00:00:09,900
Today is Thursday, and we have four items on the agenda.



In [9]:
print("First segment from each chunk:\n")

for info, result in zip(chunk_info, chunk_results):
    segments = result.get("segments", [])

    if segments:
        first = segments[0]
        start = first["start"] + info["offset_seconds"]
        end = first["end"] + info["offset_seconds"]

        print(
            f'Chunk starts at {info["offset_seconds"]:.0f}s '
            f'| [{start:.2f}s - {end:.2f}s] '
            f'{first["text"].strip()}'
        )

print("\nLast segment:")
print(all_segments[-1])

print(
    "\nPrompted and unprompted samples identical:",
    basic_transcript.text == prompted_transcript.text
)

First segment from each chunk:

Chunk starts at 0s | [0.00s - 5.80s] Good morning everyone! This is the weekly engineering sync for Project Nimbus.
Chunk starts at 60s | [60.00s - 65.96s] Yes, we need Thomas to approve the Terraform changes for the VPC peering.
Chunk starts at 120s | [120.00s - 126.16s] analyze on the slope queries and the problem is a missing composite index on the orders table
Chunk starts at 180s | [180.00s - 185.00s] Sure. We have three key results for this quarter.
Chunk starts at 240s | [240.00s - 245.80s] policy and move them to Glacier after 90 days. That would be cheap and it would
Chunk starts at 300s | [300.00s - 305.00s] And everyone, please send your greenhouse feedback by Friday.

Last segment:
{'start': 305.0, 'end': 308.0, 'text': 'Thanks everyone, see you next week!'}

Prompted and unprompted samples identical: True


In [10]:
comparison = {
    "audio_file": audio_path.name,
    "sample_duration_seconds": len(short_audio) / 1000,
    "prompt": context_prompt,
    "without_prompt": basic_transcript.text,
    "with_prompt": prompted_transcript.text,
    "identical": basic_transcript.text == prompted_transcript.text,
    "observation": (
        "The prompted and unprompted transcripts were identical "
        "for this sample. The prompt produced no observed change. "
        "Accuracy requires comparison with the original audio."
    )
}

comparison_path = project_dir / "outputs" / "comparison.json"
comparison_path.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Saved:", comparison_path.name)

Saved: comparison.json


In [11]:
report = f"""Whisper STT Implementation — Lab Report
Author: Roodabeh Seif

Implementation
I used the teacher's stt.ipynb notebook as a starting point and
added file-based transcription of {audio_path.name}.
The recording is {len(meeting_audio) / 1000:.2f} seconds long.
I tested basic transcription on its first 30 seconds, then
processed the full recording in {len(chunk_info)} chunks.

Prompted vs. unprompted transcription
I transcribed the same sample with and without a context prompt.
The prompt included Project Nimbus, Kubernetes, PostgreSQL,
Q3 OKRs, Priya, and Marcus. Both outputs were identical.
The prompt produced no observed improvement in this sample.
This does not establish perfect accuracy or mean prompts are
never useful. Accuracy must be checked against the audio.

Chunking and timestamps
I used 60-second chunks, with a shorter final chunk.
Each chunk was exported as mono, 16 kHz, 16-bit WAV.
Chunking keeps individual uploads small and supports longer
recordings. For this short recording, it demonstrated the method.
I added each chunk's starting offset to its segment timestamps.
The last transcribed segment ends at {all_segments[-1]["end"]:.2f} seconds.
Results were exported as timestamped text, SRT, and JSON.

Challenges
FFmpeg was initially unavailable to the notebook. After installation
and restarting VS Code, the notebook detected it successfully.
Fixed-duration cuts split some sentences across chunks.
The full transcript also contains suspicious technical phrases
that need verification by listening to the original recording.

Recommendations
Use clear audio and relevant names and technical vocabulary.
Prefer cuts near pauses; if using overlap, remove duplicate text.
Check transcription around chunk boundaries.
Evaluate accuracy against a manually verified reference transcript.
Preserve raw model outputs separately from any corrected version.
"""

report_path = project_dir / "lab_report.txt"
report_path.write_text(report, encoding="utf-8")

print("Saved:", report_path.name)

Saved: lab_report.txt


# What does your voice look like?

### What we import

- **`sounddevice`**: records audio from your microphone.
- **`numpy`**: holds the recording as a list of numbers.
- **`librosa`**: analyses audio and turns it into a spectrogram (a picture of pitches over time).
- **`librosa.display`**: draws that spectrogram.
- **`matplotlib.pyplot`**: creates the charts.
- **`IPython.display`** (`Audio`, `display`): plays your recording inside the notebook.

### What this code does

1. **Records 5 seconds of your voice** at 16,000 numbers per second (80,000 numbers in total).
2. **Draws your voice in 3 ways:**
   - **Waveform:** how loud your voice is at each moment.
   - **Mel spectrogram:** which pitches are present over time, adjusted to how humans hear. **This is what Whisper actually "sees."**
   - **Frequency spectrogram:** the same idea in plain frequencies (Hz), zoomed in on the speech range (0–4000 Hz).

In [ ]:
import sounddevice as sd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display

duration = 5  # seconds
sample_rate = 16000

print(f"🎤 Recording for {duration} seconds... Speak now!")
audio = sd.rec(int(duration * sample_rate), 
               samplerate=sample_rate, 
               channels=1, 
               dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")


print("This is your voice")
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

time = np.linspace(0, len(audio) / sample_rate, len(audio))
axes[0].plot(time, audio, color='blue', linewidth=0.5)
axes[0].set_title('Your Voice Waveform', fontsize=14)
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

print("Mel Spectogram") 
mel_spec = librosa.feature.melspectrogram(y=audio, sr=sample_rate, n_mels=80)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
img = librosa.display.specshow(mel_spec_db, sr=sample_rate, 
                                x_axis='time', y_axis='mel', 
                                ax=axes[1], cmap='viridis')
axes[1].set_title('Mel Spectrogram - Your Voice as a Picture!', fontsize=14)
plt.colorbar(img, ax=axes[1], format='%+2.0f dB')

print("Spectogram") 
D = librosa.stft(audio)
D_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
img2 = librosa.display.specshow(D_db, sr=sample_rate, 
                                 x_axis='time', y_axis='hz', 
                                 ax=axes[2], cmap='magma')
axes[2].set_title('Frequency Spectrogram', fontsize=14)
axes[2].set_ylim(0, 4000)  # Focus on speech frequencies
plt.colorbar(img2, ax=axes[2], format='%+2.0f dB')

plt.tight_layout()
plt.show()

# Whisper API

### What we import

- **`scipy.io.wavfile`**: turns the list of numbers into a `.wav` audio file.
- **`openai`** (`OpenAI`): connects to OpenAI so we can use Whisper.
- **`io`**: creates a file in memory, so nothing is saved to disk.
- **`os`**: reads the API key from the environment.
- **`dotenv`** (`load_dotenv`): loads the API key from the `.env` file.
- **`IPython.display`** (`Audio`, `display`): plays audio inside the notebook.

### What this code does

1. **Connects to OpenAI** using your API key.
2. **Records 5 seconds of your voice.**
3. **Packages it as a WAV file** in memory, because Whisper expects an audio file, not a list of numbers.
4. **Sends it to Whisper** and prints the text it hears.

> Run the first code cell before this one. This cell uses `sd` and `np` imported there.

In [1]:
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
import os
from IPython.display import Audio, display
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

duration = 5  # seconds
sample_rate = 16000

print(f"🎤 Recording for {duration} seconds... Speak now!")
audio = sd.rec(int(duration * sample_rate), 
               samplerate=sample_rate, 
               channels=1, 
               dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")


audio_int16 = np.int16(audio * 32767)

# Create in-memory WAV file
buffer = io.BytesIO()
wavfile.write(buffer, sample_rate, audio_int16)
buffer.seek(0)
buffer.name = "recording.wav"  # Whisper needs a filename

# Transcribe with Whisper
print("🤖 Transcribing with Whisper...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer
)

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

🎤 Recording for 5 seconds... Speak now!


NameError: name 'sd' is not defined

# Super Power: Context

Whisper only hears sounds, so it guesses wrong on rare terms and names: in our test run, *"clauses for force majeure"* came out as *"closest workforce, majority"*.
If you give it a short list of the words to expect (names, products, special terms from the client's field), it gets more of them right. There's no training: it's one extra `prompt` parameter in the same API call.

In [ ]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
from IPython.display import Audio, display, Markdown

#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Display a 30-second reading text
reading_text = """
**Read this text aloud (about 30 seconds):**

The researchers' analyses of the wind farm data showed variable wind patterns. The project 
lead will lead the team through complex permit processes. They must present the present 
findings to investors. The 3 million contract includes clauses for force majeure 
events. Dr. García-Smith's team discovered that the enzyme's pH of 7.2 was optimal. The 
AI learned to differentiate between "read" (present) and "read" (past), processing live 
feeds versus archived content. The baroque bass player stood close to the entrance, too 
close to record properly.
"""

display(Markdown(reading_text))

# Record for 30 seconds
duration = 45
sample_rate = 16000

print(f"\n🎤 Recording for {duration} seconds... Start reading now!")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")

display(Audio(audio, rate=sample_rate))


# Prepare audio
audio_int16 = np.int16(audio * 32767)
buffer = io.BytesIO()
wavfile.write(buffer, sample_rate, audio_int16)
buffer.seek(0)
buffer.name = "recording.wav"

# Transcribe without context prompt
print("🤖 Transcribing without context...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
)
text_without_context = transcript.text  # Saved for the comparison below

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

### What the `prompt` does

We send the **same recording** again, this time with a `prompt`.

- **What it is:** a few words Whisper treats as if they came *just before* the recording. It isn't an instruction like a ChatGPT prompt.
- **What it does:** when a sound is unclear, the prompt makes Whisper pick the right word and spelling. For example, *"wind farm"* instead of *"Wien farm"*, *"enzyme's"* instead of *"MCMate's"*, and *"García-Smith"* with the correct accent.
- **Why it matters:** in real projects, the prompt is where you put a client's product names, people's names and industry terms.

Compare the two outputs: without context (above) and with context (below).

In [ ]:

# Transcribe with context prompt
buffer.seek(0)  # Rewind: the buffer was already read by the previous call
print("🤖 Transcribing with context...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
    prompt = "heteronyms, homographs, lead metal, wind turbine, present tense, past tense read, pH levels, enzyme, García-Smith, force majeure, baroque, bass instrument, differentiate")
text_with_context = transcript.text  # Saved for the comparison below

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

### Before vs. after: did the prompt help?

This cell compares both transcripts with the text you actually read:

- **Word Error Rate (WER):** the share of words that are wrong, missing or extra. Lower is better.
- **Differences table:** every place where the two transcripts differ. Red is without context, green is with context.

> The prompt usually fixes the words it mentions, but it isn't magic. Look for words that got *worse* too.

In [ ]:
import re
import html
import difflib
from IPython.display import HTML, display

# The text we asked students to read (without the bold heading)
reference = reading_text.split("**")[-1].strip()

def tokenize(text):
    """Split text into words, keeping numbers like 7.2 together."""
    return re.findall(r"\d+(?:\.\d+)?|[\w'’-]+", text)

def word_error_rate(reference, hypothesis):
    """Share of words that are wrong, missing or extra compared to the reference."""
    ref = [w.lower() for w in tokenize(reference)]
    hyp = [w.lower() for w in tokenize(hypothesis)]
    # d[i][j] = fewest edits to turn the first i reference words into the first j transcript words
    d = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1):
        d[i][0] = i
    for j in range(len(hyp) + 1):
        d[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,         # word missing
                          d[i][j - 1] + 1,         # extra word
                          d[i - 1][j - 1] + cost)  # same or wrong word
    return d[len(ref)][len(hyp)] / len(ref)

wer_before = word_error_rate(reference, text_without_context)
wer_after = word_error_rate(reference, text_with_context)

# Every place where the two transcripts differ
before, after = tokenize(text_without_context), tokenize(text_with_context)
rows = ""
for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, before, after).get_opcodes():
    if op != "equal":
        old = html.escape(" ".join(before[i1:i2])) or "—"
        new = html.escape(" ".join(after[j1:j2])) or "—"
        rows += f"<tr><td style='color:#c0392b'>{old}</td><td style='color:#1e8449'>{new}</td></tr>"

display(HTML(f"""
<h3>Word Error Rate (lower is better)</h3>
<p>Without context: <b>{wer_before:.0%}</b> &nbsp;→&nbsp; With context: <b>{wer_after:.0%}</b></p>
<h3>Where the two transcripts differ</h3>
<table>
  <tr><th>Without context</th><th>With context</th></tr>
  {rows or "<tr><td colspan='2'>No differences</td></tr>"}
</table>
"""))

# Chunking

- **Why:** the Whisper API accepts files up to 25 MB, so long recordings (a 1-hour meeting) must be cut into smaller pieces and sent one by one.
- **The catch:** each piece is transcribed on its own, so a cut can split a word in half, and Whisper forgets what was said in the piece before.

In [ ]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io

#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Record longer audio
duration = 15  # seconds total
sample_rate = 16000
chunk_duration = 5  # seconds per chunk

print(f"🎤 Recording for {duration} seconds... Speak continuously!")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")

display(Audio(audio, rate=sample_rate))


# Split audio into chunks
chunk_size = chunk_duration * sample_rate
chunks = [audio[i:i+chunk_size] for i in range(0, len(audio), chunk_size)]

print(f"\n🔪 Split into {len(chunks)} chunks")

# Transcribe each chunk
all_transcripts = []

for i, chunk in enumerate(chunks):
    print(f"\n🤖 Transcribing chunk {i+1}/{len(chunks)}...")
    
    # Prepare chunk buffer
    chunk_int16 = np.int16(chunk * 32767)
    buffer = io.BytesIO()
    wavfile.write(buffer, sample_rate, chunk_int16)
    buffer.seek(0)
    buffer.name = f"chunk_{i}.wav"
    
    # Transcribe
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=buffer
    )
    
    all_transcripts.append(transcript.text)
    print(f"Chunk {i+1}: {transcript.text}")

# Combine all transcripts
print("\n📝 Complete Transcription:")
print("-" * 40)
full_text = " ".join(all_transcripts)
print(full_text)

# Time Stamps

In [ ]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
from IPython.display import Audio, display

#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

duration = 10  # seconds
sample_rate = 16000

print(f"🎤 Recording for {duration} seconds... Speak with pauses!")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")

# Play back
display(Audio(audio, rate=sample_rate))

# Prepare audio
audio_int16 = np.int16(audio * 32767)
buffer = io.BytesIO()
wavfile.write(buffer, sample_rate, audio_int16)
buffer.seek(0)
buffer.name = "recording.wav"

# Get detailed transcription with timestamps
print("\n🤖 Transcribing with timestamps...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
    response_format="verbose_json",
    timestamp_granularities=["segment", "word"]  # Get both segment and word timestamps
)

# Display formatted timestamps
print("\n📝 Transcription with timestamps:")
print("-" * 40)

# Show segments with timestamps
if hasattr(transcript, 'segments'):
    for segment in transcript.segments:
        start = segment.start
        end = segment.end
        text = segment.text
        print(f"\n[{start:.2f}s - {end:.2f}s]")
        print(f"  {text}")

# Show word-level timestamps if available
if hasattr(transcript, 'words'):
    print("\n🔤 Word-level timing:")
    print("-" * 40)
    for word in transcript.words:
        word_text = word.word
        start = word.start
        end = word.end
        print(f"{word_text:15} [{start:.2f}s - {end:.2f}s]")

# Full text and metadata
print("\n📄 Full text:")
print(transcript.text)

print(f"\n🌍 Language detected: {transcript.language}")
print(f"⏱️ Total duration: {transcript.duration:.2f}s")

# Conclusion: what we learned

### The journey from voice to text

```
Your voice → microphone → numbers (16,000 per second) → WAV file
          → Whisper: mel spectrogram → Encoder (understands) → Decoder (writes) → text
```

- **To a computer, your voice is just numbers.** The WAV file is those numbers plus a small label saying how to play them.
- **Whisper turns the numbers into a "picture" of your voice** (the mel spectrogram) and reads it. The model never hears sound. It only sees numbers.
- **Whisper is a Transformer with both halves:** the **encoder** understands the audio and the **decoder** writes the text, one token at a time.

### What each section showed

| Section | Key takeaway |
|---|---|
| **What does your voice look like?** | Sound can be shown as a waveform or a spectrogram. The mel spectrogram is what Whisper uses. |
| **Whisper API** | A few lines of code turn speech into text. |
| **Context (prompt)** | A few key words make Whisper spell names and terms correctly. It helps, but it isn't perfect, and it can't fix unclear audio. |
| **Chunking** | Long audio must be split into pieces. Each piece loses the context of the one before it. |
| **Time Stamps** | Whisper can say *when* each word was said, which is useful for subtitles, meeting notes and searching recordings. |

### For your consulting work

- **Always measure.** Use Word Error Rate on a sample of the client's real audio before promising accuracy.
- **Use the prompt** for the client's product names, people's names and industry terms.
- **Audio quality matters most.** A good microphone and a quiet room beat any clever code.
- **Voice is personal data.** Under GDPR, decide where recordings go, how long you keep them and who can access them.